[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# The Aggregation Pipeline


## What you will be able to do

Ask questions `find` cannot answer: totals by group, averages, the top three of each kind. Build a
pipeline out of `$match`, `$group`, `$sort`, `$project` and `$unwind`, and say what each stage hands
to the next. Put `$match` where an index can still be used, and see what it costs when you do not.
Say what `$unwind` does to a document whose array is empty, which is to delete it from the answer.
And read the two `$group` errors, which are both about shape rather than data.


## The idea

### The problem

`find` returns documents. It cannot count them by a field, add up a number across them, or reshape
them on the way out. Everything of that kind has to happen in Python, which means fetching every
document across the network to throw most of it away.

An aggregation moves that work to the server, and the whole of it is one list of stages.

### What a pipeline is

A list of dictionaries. Each is a stage, each takes the stream of documents from the one before and
hands a stream to the one after, and the last one's output is what you get. Stages may filter, group,
sort, reshape, or turn one document into several.

### Why order decides the cost

Only the stages at the **front** of a pipeline can use an index, because once a `$group` has run the
documents are no longer the documents in the collection: they are new ones the server made. So a
`$match` before a `$group` is an indexed lookup and the same `$match` after it is a scan of
everything the group produced.

### Where this shows up

Every report, every dashboard, every "how many of each" question. Also every migration that has to
find the documents with some property no index covers.

### What this notebook covers

`$match`, `$group` and its accumulators, `$sort`, `$limit`, `$project`. `$unwind`, and the documents
it removes. Stage order, measured. Then the two `$group` shape errors, the `$unwind` that loses
rows, and the `$project` that changes a type without telling you.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

shop.baskets.drop()
shop.baskets.insert_many([
    {"_id": 1, "items": ["a", "b"]},
    {"_id": 2, "items": []},                       # an empty basket
    {"_id": 3},                                    # no items field at all
])

rows = list(shop.baskets.aggregate([{"$unwind": "$items"}]))

print("documents in:", shop.baskets.count_documents({}))
print("rows out:    ", len(rows), "from baskets", [row["_id"] for row in rows])
print("baskets 2 and 3 are gone, and nothing said so")
client.close()
```

```
documents in: 3
rows out:     2 from baskets [1, 1]
baskets 2 and 3 are gone, and nothing said so
```

Three baskets went in and two rows came out, both from the same basket. `$unwind` turns one document
into one per array element, and a document with no elements produces none. A report counting baskets
after an `$unwind` is counting something else.


## Setup

Seven imports, MongoDB, the boot cell, and three helpers.

- `pymongo` is the driver, and `time` is there for the boot cell's readiness loop
- `subprocess` and `os` install and start the server, `sys` names this Python
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

`work_done` runs `explain` over a pipeline and reports what the server had to read, which is how the
stage order section is measured rather than asserted. `build_baskets` makes the three baskets from
the first look. `failed` prints a failure's message without the parts that change between runs.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """A failure's real message, without the cluster time that changes every run."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def work_done(pipeline, collection="products"):
    """What the server had to read to run a pipeline, which is what stage order decides."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        explained = shop.command("explain",
                                 {"aggregate": collection, "pipeline": pipeline, "cursor": {}},
                                 verbosity="executionStats")
        stats = explained.get("executionStats")
        if stats is None:                                           # a pipeline with stages after it
            stats = explained["stages"][0]["$cursor"]["executionStats"]
        return {"documents": stats["totalDocsExamined"], "index keys": stats["totalKeysExamined"]}


def build_baskets():
    """Three baskets: one with items, one with an empty array, one with no such field."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        shop.baskets.drop()
        shop.baskets.insert_many([{"_id": 1, "owner": "ana", "items": ["a", "b"]},
                                  {"_id": 2, "owner": "bo", "items": []},
                                  {"_id": 3, "owner": "cy"}])
        return shop.baskets.count_documents({})


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print("baskets:", build_baskets())
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
baskets: 3
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### $group, and the _id that is the grouping key


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

by_kind = list(shop.products.aggregate([
    {"$group": {"_id": "$kind", "n": {"$sum": 1}}},
    {"$sort": {"_id": 1}},
]))
for row in by_kind:
    print(f"  {row['_id']:9} {row['n']}")


  cable     100
  keyboard  100
  laptop    100
  monitor   100
  mouse     100


`_id` in a `$group` is not a document id. It is the expression to group by, and it becomes the `_id`
of each output document. `"$kind"` with a dollar sign means "the value of the field `kind`", which
is how every expression in an aggregation refers to a field.

Grouping by `None` gives one row for everything:


In [3]:
print(list(shop.products.aggregate([
    {"$group": {"_id": None,
                "products": {"$sum": 1},
                "stock": {"$sum": "$stock"},
                "dearest": {"$max": "$price"},
                "average": {"$avg": "$price"}}},
    {"$project": {"_id": 0, "products": 1, "stock": 1, "dearest": 1,
                  "average": {"$round": ["$average", 2]}}},
])))


[{'products': 500, 'stock': 99019, 'dearest': 1999.73, 'average': 1005.54}]


`$sum`, `$max`, `$min`, `$avg`, `$push` and `$addToSet` are the accumulators. `$sum: 1` counts,
because it adds one per document, and `$sum: "$stock"` totals a field.

### $match, $sort, $limit, $project

The rest of a pipeline is the parts you already know from `find`, in stage form:


In [4]:
dearest_laptops = list(shop.products.aggregate([
    {"$match": {"kind": "laptop"}},
    {"$sort": {"price": -1}},
    {"$limit": 3},
    {"$project": {"_id": 0, "name": 1, "price": 1}},
]))
for row in dearest_laptops:
    print(f"  {row['price']:8.2f}  {row['name']}")


   1956.40  Belden laptop 465
   1933.27  Aster laptop 20
   1931.82  Aster laptop 400


`$project` is the projection from **find and find_one**, with one addition that matters: it can
compute. A field set to an expression rather than to `1` is a new field built from the others.


In [5]:
print(list(shop.products.aggregate([
    {"$match": {"_id": {"$lt": 3}}},
    {"$project": {"_id": 0,
                  "name": 1,
                  "area": {"$multiply": ["$size.w", "$size.h"]},
                  "cheap": {"$lt": ["$price", 500]}}},
])))


[{'name': 'Aster laptop 0', 'area': 714, 'cheap': False}, {'name': 'Belden monitor 1', 'area': 230, 'cheap': False}, {'name': 'Corvid keyboard 2', 'area': 192, 'cheap': False}]


### Two stages at once, by group

The shape that answers "the top of each kind" in one query:


In [6]:
top_of_each = list(shop.products.aggregate([
    {"$sort": {"price": -1}},
    {"$group": {"_id": "$kind",
                "dearest": {"$first": "$name"},
                "price": {"$first": "$price"}}},
    {"$sort": {"_id": 1}},
]))
for row in top_of_each:
    print(f"  {row['_id']:9} {row['price']:8.2f}  {row['dearest']}")


  cable      1991.32  Dalgo cable 499
  keyboard   1950.73  Dalgo keyboard 447
  laptop     1956.40  Belden laptop 465
  monitor    1999.73  Corvid monitor 466
  mouse      1989.91  Corvid mouse 138


`$first` takes the first document of each group **in the order the group received them**, which is
why the `$sort` comes before the `$group` and not after. A `$sort` after the group would sort the
groups, not their contents.

That is the whole trick and it is worth stating plainly: a pipeline has no idea what you meant, only
what order you put the stages in.

### $unwind

One document per array element:


In [7]:
print("the baskets:")
for basket in shop.baskets.find().sort("_id"):
    print("  ", basket)

print()
print("after $unwind:")
for row in shop.baskets.aggregate([{"$unwind": "$items"}]):
    print("  ", row)


the baskets:
   {'_id': 1, 'owner': 'ana', 'items': ['a', 'b']}
   {'_id': 2, 'owner': 'bo', 'items': []}
   {'_id': 3, 'owner': 'cy'}

after $unwind:
   {'_id': 1, 'owner': 'ana', 'items': 'a'}
   {'_id': 1, 'owner': 'ana', 'items': 'b'}


Basket 1 became two rows. Baskets 2 and 3 became nothing, because an empty array has no elements to
make a row from and a missing field is the same as an empty one here.

`preserveNullAndEmptyArrays` keeps them, with the field absent:


In [8]:
for row in shop.baskets.aggregate([
        {"$unwind": {"path": "$items", "preserveNullAndEmptyArrays": True}}]):
    print("  ", row)


   {'_id': 1, 'owner': 'ana', 'items': 'a'}
   {'_id': 1, 'owner': 'ana', 'items': 'b'}
   {'_id': 2, 'owner': 'bo'}
   {'_id': 3, 'owner': 'cy'}


Which you want depends on the question. "How many items were bought" wants the default. "How many
baskets were there, and what was in them" wants the preserving form, because a basket with nothing
in it is still a basket.

### Where $match goes

`$match` at the front can use an index. The same `$match` after a `$group` cannot:


In [9]:
shop.products.create_index([("kind", 1), ("price", 1)], name="kind_price")

early = [{"$match": {"kind": "laptop"}},
         {"$group": {"_id": "$maker", "n": {"$sum": 1}}}]

late = [{"$group": {"_id": "$maker", "n": {"$sum": 1}, "kinds": {"$addToSet": "$kind"}}},
        {"$match": {"kinds": "laptop"}}]

print("$match then $group:", work_done(early))
print("$group then $match:", work_done(late))


$match then $group: {'documents': 100, 'index keys': 100}
$group then $match: {'documents': 500, 'index keys': 0}


A hundred documents against five hundred, and a hundred index keys against none. The second pipeline
read the whole collection, because by the time the `$match` ran, the documents it was filtering were
the groups rather than the products, and no index covers those.

Over five hundred documents that is a rounding error. Over the two hundred thousand of **Indexes**
it is the difference between a report and a timeout.

The rule: put every `$match` you can at the front, and anything narrowing the input before anything
that reshapes it.

### A sales report, finished


In [10]:
def report(shop, kind=None, cheapest_first=False):
    """One pipeline: filter first, then group, then present."""
    pipeline = []
    if kind is not None:
        pipeline.append({"$match": {"kind": kind}})                 # first, so an index can help

    pipeline += [
        {"$group": {"_id": "$maker",
                    "products": {"$sum": 1},
                    "stock": {"$sum": "$stock"},
                    "average": {"$avg": "$price"}}},
        {"$project": {"_id": 0,
                      "maker": "$_id",
                      "products": 1,
                      "stock": 1,
                      "average": {"$round": ["$average", 2]}}},
        {"$sort": {"average": 1 if cheapest_first else -1}},
    ]
    return list(shop.products.aggregate(pipeline))


print("every kind, dearest maker first:")
for row in report(shop):
    print(f"  {row['maker']:8} {row['products']:4} products  {row['stock']:6} in stock  "
          f"average {row['average']:8.2f}")

print()
print("laptops only, cheapest maker first:")
for row in report(shop, kind="laptop", cheapest_first=True):
    print(f"  {row['maker']:8} {row['products']:4} products  average {row['average']:8.2f}")


every kind, dearest maker first:
  Aster     125 products   24145 in stock  average  1058.70
  Belden    125 products   24862 in stock  average  1008.70
  Dalgo     125 products   24971 in stock  average  1006.07
  Corvid    125 products   25041 in stock  average   948.67

laptops only, cheapest maker first:
  Dalgo      25 products  average   943.79
  Corvid     25 products  average   950.86
  Belden     25 products  average  1004.50
  Aster      25 products  average  1159.74


The `$match` is added first when there is one, so the index does its work before anything else
happens. `$project` renames `_id` back to `maker`, because a grouping key called `_id` in the output
is confusing to everybody downstream. And the final `$sort` is on a field the `$project` computed,
which is allowed precisely because it comes after it.

### Where each part came from

| In `report` | What it relies on | The section that showed it |
|---|---|---|
| `$match` added first | only the front of a pipeline uses an index | Where $match goes |
| `{"_id": "$maker"}` | the grouping key, named `_id` | $group, and the _id |
| `$sum` and `$avg` | accumulators over a group | $group, and the _id |
| `"maker": "$_id"` | `$project` renaming a field | $match, $sort, $limit, $project |
| `$round` | `$project` computing rather than selecting | $match, $sort, $limit, $project |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/09-the-aggregation-pipeline-solutions.ipynb).

**1.** Count the products of each maker.


In [11]:
# your code here


**2.** Find the total stock and the average price across the whole collection.


In [12]:
# your code here


**3.** Print the cheapest product of each kind.


In [13]:
# your code here


**4.** Count how many times each tag is used, with `$unwind`.


In [14]:
# your code here


**5.** Show that `$unwind` drops the empty baskets and `preserveNullAndEmptyArrays` keeps them.


In [15]:
# your code here


**6.** Compare the work done by `$match` before and after a `$group`.


In [16]:
# your code here


## Common errors

### pymongo.errors.OperationFailure: a group specification must include an _id


In [17]:
try:
    list(shop.products.aggregate([{"$group": {"n": {"$sum": 1}}}]))
except pymongo.errors.OperationFailure as error:
    print(failed(error), "| codeName:", error.details["codeName"])


OperationFailure: a group specification must include an _id | codeName: Location15955


`$group` always needs to know what to group by, and there is no default. Counting everything is
`{"_id": None}`, which is explicit rather than omitted.


In [18]:
print(list(shop.products.aggregate([{"$group": {"_id": None, "n": {"$sum": 1}}}])))


[{'_id': None, 'n': 500}]


### pymongo.errors.OperationFailure: The field 'total' must be an accumulator object


In [19]:
try:
    list(shop.products.aggregate([{"$group": {"_id": "$kind", "total": 1}}]))
except pymongo.errors.OperationFailure as error:
    print(failed(error), "| codeName:", error.details["codeName"])


OperationFailure: The field 'total' must be an accumulator object | codeName: Location40234


Inside a `$group`, every field except `_id` must say **how to combine** the documents in the group,
not what value to take. `1` is a value, and there is no rule for combining a hundred documents into
the number one.

This is the error you get from writing `$group` with the habits of `$project`:


In [20]:
print("counting:", list(shop.products.aggregate([
    {"$group": {"_id": "$kind", "total": {"$sum": 1}}}, {"$sort": {"_id": 1}}, {"$limit": 2}])))
print("or keeping one value per group:", list(shop.products.aggregate([
    {"$group": {"_id": "$kind", "an_example": {"$first": "$name"}}},
    {"$sort": {"_id": 1}}, {"$limit": 2}])))


counting: [{'_id': 'cable', 'total': 100}, {'_id': 'keyboard', 'total': 100}]
or keeping one value per group: [{'_id': 'cable', 'an_example': 'Dalgo cable 399'}, {'_id': 'keyboard', 'an_example': 'Belden keyboard 277'}]


### No error: the rows $unwind removed


In [21]:
build_baskets()

owners = list(shop.baskets.aggregate([
    {"$unwind": "$items"},
    {"$group": {"_id": "$owner"}},
]))
print("owners with a basket, after $unwind:", sorted(row["_id"] for row in owners))
print("owners with a basket, in fact:      ",
      sorted(basket["owner"] for basket in shop.baskets.find()))


owners with a basket, after $unwind: ['ana']
owners with a basket, in fact:       ['ana', 'bo', 'cy']


Two owners lost. A report built this way undercounts by exactly the number of documents whose array
was empty, which in real data is usually the interesting ones: the customers who bought nothing, the
orders with no lines, the posts with no comments.

Preserve them when the question is about the documents rather than the elements:


In [22]:
owners = list(shop.baskets.aggregate([
    {"$unwind": {"path": "$items", "preserveNullAndEmptyArrays": True}},
    {"$group": {"_id": "$owner", "items": {"$sum": {"$cond": [{"$ifNull": ["$items", False]},
                                                              1, 0]}}}},
    {"$sort": {"_id": 1}},
]))
print("every owner, with a count that can be zero:", owners)


every owner, with a count that can be zero: [{'_id': 'ana', 'items': 2}, {'_id': 'bo', 'items': 0}, {'_id': 'cy', 'items': 0}]


### No error: $group finding more memory than you expected

A `$group` holds one entry per distinct key while it runs, and that is the number to think about:


In [23]:
one_per_product = [{"$group": {"_id": "$_id", "name": {"$first": "$name"}}}]
five_groups = [{"$group": {"_id": "$kind", "n": {"$sum": 1}}}]

print("grouping by kind:", len(list(shop.products.aggregate(five_groups))), "groups")
print("grouping by _id: ", len(list(shop.products.aggregate(one_per_product))), "groups")
print()
print("both ran, and one of them built a group for every document in the collection")


grouping by kind: 5 groups
grouping by _id:  500 groups

both ran, and one of them built a group for every document in the collection


Since MongoDB 6.0 a stage that runs out of memory spills to disk rather than failing, so the second
pipeline succeeds where older versions answered
`Exceeded memory limit for $group, which is 100MB`. That is a kindness and it is also why the
failure no longer warns you: the query gets slower instead of stopping, and the only sign is a
report that used to take a second.

`allowDiskUse=False` turns the kindness off, which is worth doing in a test if you want to be told:


In [24]:
print("with spilling allowed:", len(list(shop.products.aggregate(one_per_product,
                                                                 allowDiskUse=True))), "groups")
print("without it:           ", len(list(shop.products.aggregate(one_per_product,
                                                                 allowDiskUse=False))), "groups")
print()
print("this collection is small enough that neither needs the disk; a large one would")


with spilling allowed: 500 groups
without it:            500 groups

this collection is small enough that neither needs the disk; a large one would


In [25]:
shop.baskets.drop()
client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- A pipeline is a list of stages. Each takes the stream from the one before, and the order is the
  whole of the semantics.
- `$group` needs an `_id`, which is the expression to group by, and every other field must be an
  accumulator: `$sum`, `$avg`, `$max`, `$min`, `$first`, `$push`, `$addToSet`.
- `"$field"` with a dollar sign is how an expression refers to a field's value.
- `$project` selects and also computes, so it can rename `_id` back to something readable and build
  new fields from the others.
- `$first` takes the first document of a group in the order the group received them, so the `$sort`
  that decides "first" goes **before** the `$group`.
- `$unwind` makes one row per array element and **removes** documents whose array is empty or
  missing, unless `preserveNullAndEmptyArrays` is set.
- Only stages at the front of a pipeline can use an index. `$match` before `$group` is a lookup and
  after it is a scan.
- Since MongoDB 6.0 a stage over its memory limit spills to disk instead of failing, so the symptom
  is slowness rather than an error.


## What is next

**Modeling Without Joins** is the design question underneath all of this: embed until the document
will not fit, what happens when it will not, and the `$lookup` that scans the other collection once
per input document because nobody indexed the field it joins on.


---

&#8592; **Previous:** [Indexes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/08-indexes.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Modeling Without Joins](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/10-modeling-without-joins.ipynb) &#8594;
